# GSG tracer concentration example

This notebook updates the older `GSG_tracer_concentration.ipynb` workflow for the current ECLlib GSG API.
It writes PROP-style `.GSG` files with `GSGProperty` and `GSGFile.write_prop`, then reads them back with `GSGFile(...).read()`.

The default output file uses an `_EXAMPLE` suffix in the configured case folder so `GSGFile` can discover the adjacent `.afi`/`.ixf` files.

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from ECLlib import GSGFile, GSGProperty

## Configure a case

Set `case_root` to the case stem, without extension. For an INTERSECT case this is normally the `.afi` stem. The default below uses a Volve-style case name; change the path to any case folder containing the grid `.GSG`, `.afi`, and `.ixf` files.

In [ ]:
case_root = Path.home() / "cases/volve/VOLVE_2016"
output_dir = case_root.parent
output_file = output_dir / f"{case_root.name}_TRACER_CONCENTRATION_EXAMPLE.GSG"
overwrite_existing = True

print(f"Case root : {case_root}")
print(f"Output    : {output_file}")

In [ ]:
def describe_properties(properties):
    """Print compact metadata for GSG properties."""
    for prop in properties:
        print(
            f"alias={prop.alias:8s} encoding={prop.encoding:4s} dtype={prop.values.dtype} "
            f"shape={prop.values.shape} names={prop.names} roles={prop.roles}"
        )


def plot_i_slice(prop, i=None, aspect=5):
    """Plot one I-slice from a 3D property array."""
    if prop.values.ndim != 3:
        raise ValueError(f"Expected a 3D property, got shape {prop.values.shape}")
    i = prop.values.shape[0] // 2 if i is None else min(int(i), prop.values.shape[0] - 1)
    plt.figure(figsize=(8, 4))
    plt.title(f"{prop.names[0]} ({prop.alias}), i={i}")
    plt.imshow(prop.values[i, :, :].T, origin="upper", aspect=aspect, interpolation="none")
    plt.colorbar(orientation="horizontal")
    plt.show()

## Write one constant tracer concentration

This creates a single `TRACER_CONCENTRATION["CL"]` property. Values are stored as `float32` and written as a full array.

In [ ]:
dim = GSGFile(case_root.with_suffix(".GSG")).dim()
print(f"Dimensions: {dim}")

cl = np.ones(dim, dtype=np.float32)
cl_property = GSGProperty.from_array(
    'TRACER_CONCENTRATION["CL"]',
    "TRCL",
    cl,
    role="p",
    encoding="full",
)

output_dir.mkdir(parents=True, exist_ok=True)
GSGFile.write_prop(output_file, cl_property, overwrite=overwrite_existing)
print(f"Wrote {output_file}")

In [ ]:
properties = GSGFile(output_file).read()
describe_properties(properties)
plot_i_slice(properties[0], i=dim[0] // 2)

## Write separate lower and upper tracer regions

This variant writes two tracer properties to the same GSG file. Adjust `split_k` to match the layer split you want.

In [ ]:
split_k = dim[2] // 2

lower = np.zeros(dim, dtype=np.float32)
lower[:, :, :split_k] = 1.0

upper = np.zeros(dim, dtype=np.float32)
upper[:, :, split_k:] = 1.0

tracer_properties = (
    GSGProperty.from_array('TRACER_CONCENTRATION["LOWER"]', "TRLOW", lower, role="p"),
    GSGProperty.from_array('TRACER_CONCENTRATION["UPPER"]', "TRUP", upper, role="p"),
)

GSGFile.write_prop(output_file, *tracer_properties, overwrite=overwrite_existing)
print(f"Wrote {len(tracer_properties)} tracer properties to {output_file}")

In [ ]:
properties = GSGFile(output_file).read()
describe_properties(properties)
for prop in properties:
    plot_i_slice(prop, i=dim[0] // 2)

## Inspect an existing property file

The same reader API can inspect Petrel/INTERSECT PROP GSG files. This example reads `PERM_I` beside the configured case root if the file exists.

In [ ]:
property_file = case_root.with_name(f"{case_root.name}_PERM_I.GSG")
if not property_file.exists():
    print(f"No example property file found: {property_file}")
else:
    perm_properties = GSGFile(property_file).read()
    describe_properties(perm_properties)
    plot_i_slice(perm_properties[0], i=dim[0] // 2)

## Model_From_IXF variant

Point `model_root` at a Model_From_IXF case stem and reuse the same writer calls. The example runs only when a grid `.GSG` file exists beside the case.

In [ ]:
model_root = Path.home() / "cases/Model_From_IXF/Model_From_IXF"
model_grid = model_root.with_suffix(".GSG")
model_output = model_root.parent / "Model_From_IXF_TRACER_CONCENTRATION_EXAMPLE.GSG"

if not model_grid.exists():
    print(f"No grid GSG file found for optional example: {model_grid}")
else:
    model_dim = GSGFile(model_grid).dim()
    model_cl = np.ones(model_dim, dtype=np.float32)
    model_property = GSGProperty.from_array(
        "TRACER_CONCENTRATION['CL']",
        "TR",
        model_cl,
        role="p",
    )
    GSGFile.write_prop(model_output, model_property, overwrite=overwrite_existing)
    describe_properties(GSGFile(model_output).read())
    print(f"Wrote {model_output}")